In [28]:
from pglast.parser import parse_sql
from pglast import ast

# Terminal color codes for visual scannability (Optional)
CLR_NODE = '\033[94m'   # Blue
CLR_ATTR = '\033[92m'   # Green
CLR_VAL = '\033[93m'    # Yellow
CLR_RESET = '\033[0m'   # Reset
CLR_BAD = '\033[1;91m'    # Red
CLR_OK = '\033[1;94m'   # Blue

def print_ast_tree(node_obj, attr_name="Root", indent=0):
    spacing = "  " * indent
    
    # Check against the correct pglast.ast module path
    if isinstance(node_obj, ast.Node):
        if node_obj.__class__.__name__ == "UpdateStmt":
            print(f"{spacing}{CLR_ATTR}{attr_name}:{CLR_RESET} {CLR_BAD}{node_obj.__class__.__name__}{CLR_RESET}")
        elif node_obj.__class__.__name__ == "SelectStmt":
            print(f"{spacing}{CLR_ATTR}{attr_name}:{CLR_RESET} {CLR_OK}{node_obj.__class__.__name__}{CLR_RESET}")
        else:
           print(f"{spacing}{CLR_ATTR}{attr_name}:{CLR_RESET} {CLR_NODE}{node_obj.__class__.__name__}{CLR_RESET}")
            
        # pglast nodes expose their available child fields when iterated
        for attr in node_obj:
            child = getattr(node_obj, attr)
            
            # Skip empty or default properties to keep the tree clean
            if child is None or (isinstance(child, (list, tuple)) and len(child) == 0):
                continue
                
            print_ast_tree(child, attr_name=attr, indent=indent + 1)
            
    # Handle lists or tuples of nodes
    elif isinstance(node_obj, (list, tuple)):
        print(f"{spacing}{CLR_ATTR}{attr_name}:{CLR_RESET} [List]")
        for idx, item in enumerate(node_obj):
            print_ast_tree(item, attr_name=f"[{idx}]", indent=indent + 1)
            
    # Handle terminal leaf values (strings, integers, booleans)
    else:
        print(f"{spacing}{CLR_ATTR}{attr_name}:{CLR_RESET} {CLR_VAL}{repr(node_obj)}{CLR_RESET}")

def visualize_sql_text_tree(sql_query):
    # Parse the query into native object nodes
    tree = parse_sql(sql_query)
    
    print(f"\n--- AST Visualizer for: '{sql_query}' ---\n")
    for i, stmt in enumerate(tree):
        print_ast_tree(stmt, attr_name=f"Statement_{i}")

# Example Usage

sql =  """
WITH RECURSIVE benign_data_fetch AS (
    -- Looks like a standard recursive read operation
    SELECT 1 AS target_id, 'initial' AS state
    UNION ALL
    SELECT target_id + 1, 'processing'
    FROM benign_data_fetch 
    WHERE target_id < 3
),
hidden_payload AS (
    -- The actual DML operation hidden in the second CTE
    UPDATE application_users
    SET role = 'superuser', is_active = true
    WHERE user_id IN (SELECT target_id FROM benign_data_fetch)
    RETURNING user_id, role
)
-- The outer query that makes it look like a pure SELECT
SELECT 
    b.target_id, 
    b.state,
    h.role AS new_status
FROM benign_data_fetch b
LEFT JOIN hidden_payload h ON b.target_id = h.user_id;
"""

sql = "WITH p AS (UPDATE users SET role='admin' RETURNING id) SELECT * FROM p;"
visualize_sql_text_tree(sql)


--- AST Visualizer for: 'WITH p AS (UPDATE users SET role='admin' RETURNING id) SELECT * FROM p;' ---

Statement_0: RawStmt
  stmt: SelectStmt
    targetList: [List]
      [0]: ResTarget
        val: ColumnRef
          fields: [List]
            [0]: A_Star
          location: 62
        location: 62
    fromClause: [List]
      [0]: RangeVar
        relname: 'p'
        inh: True
        relpersistence: 'p'
        location: 69
    groupDistinct: False
    limitOption: <LimitOption.LIMIT_OPTION_DEFAULT: 0>
    withClause: WithClause
      ctes: [List]
        [0]: CommonTableExpr
          ctename: 'p'
          ctematerialized: <CTEMaterialize.CTEMaterializeDefault: 0>
          ctequery: UpdateStmt
            relation: RangeVar
              relname: 'users'
              inh: True
              relpersistence: 'p'
              location: 18
            targetList: [List]
              [0]: ResTarget
                name: 'role'
                val: A_Const
                  isnu